# MENTORA — Train M3: Performance Trajectory Predictor (LSTM)

Do this one first — no HuggingFace dependency, trains from scratch on the synthetic score sequences from Phase 3. Target: **val MAE <= 5**.

## 0. Shared setup — mount Drive, confirm GPU, install deps
Run these three cells first in every notebook (per Phase 4 §0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

# NOTE: upload the repo's datasets/ folder to Google Drive at this path first
# (mentora_data/{raw,processed,labeled}/...) — see datasets/README.md.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121  # Colab GPU wheel — do NOT reuse the CPU wheel from the backend's laptop plan
!pip install -q transformers peft accelerate sentence-transformers datasets evaluate seqeval spacy sacrebleu rouge_score

## 1. Load and window the data

In [ ]:
import pandas as pd, numpy as np, torch
from torch.utils.data import Dataset, DataLoader

df = pd.read_csv(f'{DATA}/processed/m3/score_histories_no_label.csv')

SEQ_LEN = 5   # feed 5 past sessions, predict the next score
class TrajectoryDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN):
        self.samples = []
        for student_id, group in df.groupby('student_id'):
            scores = group.sort_values('session_number')['score'].values
            for i in range(len(scores) - seq_len):
                x = scores[i:i+seq_len]
                y = scores[i+seq_len]
                self.samples.append((x, y))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.float32).unsqueeze(-1), torch.tensor(y, dtype=torch.float32)

full_ds = TrajectoryDataset(df)
n_val = int(0.15 * len(full_ds))
train_ds, val_ds = torch.utils.data.random_split(full_ds, [len(full_ds) - n_val, n_val])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
print(f'{len(train_ds)} train windows, {len(val_ds)} val windows')

## 2. Model definition

In [ ]:
import torch.nn as nn

class TrajectoryLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TrajectoryLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

## 3. Training loop — checkpoints to Drive every epoch, resumable

In [ ]:
CKPT_PATH = f'{MODELS}/model3_trajectory_predictor/checkpoint.pt'
start_epoch = 0

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {start_epoch}')

N_EPOCHS = 30
for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_abs_errors = []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_abs_errors.extend((pred - y).abs().cpu().tolist())
    val_mae = np.mean(val_abs_errors)

    print(f'epoch {epoch+1}/{N_EPOCHS} - train_loss {train_loss:.3f} - val_MAE {val_mae:.3f}')

    torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'epoch': epoch},
               CKPT_PATH)

    if val_mae <= 5.0:
        print(f'Target MAE <= 5 reached at epoch {epoch+1} (val_MAE={val_mae:.2f}) - can stop here')
        break

## 4. Save final model

In [ ]:
torch.save(model.state_dict(), f'{MODELS}/model3_trajectory_predictor/lstm_final.pt')
print('M3 done. Target: MAE <= 5. Achieved:', val_mae)

## Target metric: MAE <= 5

If validation MAE plateaus above that:
- Check archetype balance from Phase 3's validation checklist
- Try `hidden_size=256` or a 3rd LSTM layer
- Increase `SEQ_LEN` to 7-8 if most students have enough sessions